# S03E01 — Evaluation

**Zadanie:** Znaleźć anomalie w odczytach sensorów elektrowni.

Anomalia = jedno z poniższych:
- wartość poza zakresem normy dla aktywnego czujnika
- operator twierdzi że jest OK, ale dane są złe
- operator twierdzi że są błędy, ale dane są OK
- czujnik zwraca wartości dla pola, którego nie powinien (np. temperature-only zwraca pressure)

**Techniki:** analiza danych, LLM jako ewaluator notatek operatora, batch processing

In [ ]:
import os, json, zipfile, io, requests
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
SENSORS_URL = "https://hub.ag3nts.org/dane/sensors.zip"

# Zakresy normy dla aktywnych sensorów
RANGES = {
    "temperature_K":      (553, 873),
    "pressure_bar":       (60, 160),
    "water_level_meters": (5.0, 15.0),
    "voltage_supply_v":   (229.0, 231.0),
    "humidity_percent":   (40.0, 80.0),
}

# Mapowanie: jeśli sensor_type zawiera słowo kluczowe → które pola są aktywne
TYPE_TO_FIELDS = {
    "temperature": ["temperature_K"],
    "pressure":    ["pressure_bar"],
    "water":       ["water_level_meters"],
    "voltage":     ["voltage_supply_v"],
    "humidity":    ["humidity_percent"],
}

print("Konfiguracja OK.")

In [ ]:
# Pobierz i rozpakuj pliki sensorów
resp = requests.get(SENSORS_URL)
resp.raise_for_status()

sensors = {}  # id -> dict
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    for name in zf.namelist():
        if name.endswith(".json"):
            sensor_id = name.replace(".json", "").split("/")[-1]
            data = json.loads(zf.read(name))
            sensors[sensor_id] = data

print(f"Załadowano {len(sensors)} plików sensorów.")
# Przykładowy odczyt
sample_id = list(sensors.keys())[0]
print(f"Przykład [{sample_id}]:", json.dumps(sensors[sample_id], indent=2))

In [ ]:
def get_active_fields(sensor_type: str) -> list[str]:
    """Zwraca listę pól, które POWINNY być aktywne dla danego sensor_type."""
    parts = [p.strip().lower() for p in sensor_type.split("/")]
    active = []
    for part in parts:
        for key, fields in TYPE_TO_FIELDS.items():
            if key in part:
                active.extend(fields)
    return list(set(active))


def check_numeric_anomaly(sensor_id: str, data: dict) -> list[str]:
    """Wykryj anomalie numeryczne: wartości poza zakresem lub niewłaściwe pola."""
    reasons = []
    sensor_type = data.get("sensor_type", "")
    active_fields = get_active_fields(sensor_type)

    for field, (lo, hi) in RANGES.items():
        value = data.get(field, 0)
        is_active = field in active_fields

        if is_active:
            if value == 0:
                reasons.append(f"Aktywne pole {field} ma wartość 0")
            elif not (lo <= value <= hi):
                reasons.append(f"{field}={value} poza zakresem [{lo}, {hi}]")
        else:
            # Nieaktywne pole — powinno być 0
            if value != 0:
                reasons.append(f"Nieaktywne pole {field}={value} (powinno być 0)")

    return reasons


# Wykryj anomalie numeryczne
numeric_anomalies = {}
for sid, data in sensors.items():
    reasons = check_numeric_anomaly(sid, data)
    if reasons:
        numeric_anomalies[sid] = reasons

print(f"Anomalie numeryczne: {len(numeric_anomalies)}")
for sid, r in list(numeric_anomalies.items())[:5]:
    print(f"  [{sid}]: {r}")

In [ ]:
# Wykryj anomalie w notatkach operatora (LLM)
# Sprawdź czy notatka zgadza się ze stanem numerycznym

def check_note_anomaly_batch(batch: list[tuple[str, dict, list[str]]]) -> list[str]:
    """Sprawdź notatki operatora dla batcha sensorów. Zwraca listę ID z anomaliami notatek."""
    lines = []
    for sid, data, num_reasons in batch:
        numeric_ok = len(num_reasons) == 0
        lines.append(
            f'ID={sid} numeric_ok={numeric_ok} note="{data.get("operator_notes", "")}"'
        )

    prompt = f"""Przeanalizuj notatki operatora dla sensorów. Dla każdego sensora oceń:
czy notatka operatora jest SPÓJNA z danymi numerycznymi (numeric_ok).

Anomalia notatki = operator twierdzi że jest OK ale numeric_ok=False,
LUB operator twierdzi że są błędy ale numeric_ok=True.

Dane:
{chr(10).join(lines)}

Odpowiedz TYLKO listą JSON z ID sensorów mających anomalię notatki:
["0001", "0042", ...]
Jeśli brak anomalii notatek — zwróć []"""

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    text = resp.content[0].text.strip()
    try:
        start = text.find("[")
        end   = text.rfind("]") + 1
        return json.loads(text[start:end])
    except Exception:
        return []


# Przetwarzaj w batchach po 50
BATCH_SIZE = 50
all_items  = list(sensors.items())
note_anomalies = set()

for i in range(0, len(all_items), BATCH_SIZE):
    batch = [
        (sid, data, numeric_anomalies.get(sid, []))
        for sid, data in all_items[i:i+BATCH_SIZE]
    ]
    result = check_note_anomaly_batch(batch)
    note_anomalies.update(result)
    print(f"  Batch {i//BATCH_SIZE + 1}: {len(result)} anomalii notatek")

print(f"\nAnomalie notatek łącznie: {len(note_anomalies)}")

In [ ]:
# Połącz wszystkie anomalie i wyślij
all_anomalies = sorted(set(numeric_anomalies.keys()) | note_anomalies)
print(f"Łączna liczba anomalii: {len(all_anomalies)}")
print("IDs:", all_anomalies[:20], "...")

result = requests.post(VERIFY_URL, json={
    "apikey": API_KEY,
    "task": "evaluation",
    "answer": {"recheck": all_anomalies}
}).json()

print("\nOdpowiedź Hub-u:", result)